# Building a Multi-Agent DevOps System

By the end of this notebook you'll have a 3-agent pipeline that **reads production logs**, **searches for solutions online**, and **writes a step-by-step remediation plan** — all automatically.

We build it one concept at a time.

| Step | Concept | What It Unlocks |
|------|---------|----------------|
| 1 | **Agent** | A specialized AI worker with a role and goal |
| 2 | **Task** | A specific job you assign to an agent |
| 3 | **Tool** | Gives the agent real-world capabilities (read files, search web) |
| 4 | **Parameters** | Controls for iteration limits, rate limiting, timeouts |
| 5 | **Context** | Output from one task flows into the next |
| 6 | **Crew** | Orchestrates multiple agents working together |

In [ ]:
import os

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

---

## Step 1: Your First Agent

An **Agent** is a specialized AI worker. You define *who* it is (role, backstory) and *what* it's trying to do (goal). The LLM does the rest.

In [ ]:
log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

print(f"Agent created: {log_analyzer.role}")

---

## Step 2: Your First Task

An agent without a job does nothing. A **Task** tells the agent exactly what to do and what to produce. We'll pass a hardcoded log snippet directly in the task description.

In [ ]:
LOG_INPUT = """
[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment
[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
[2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints
[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated
"""

analyze_task = Task(
    description=f"Analyze the following log data to identify issues:\n{LOG_INPUT}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

crew = Crew(
    agents=[log_analyzer],
    tasks=[analyze_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()
print(result.raw)

One agent, one task, one crew — and it produced a full analysis. But we hardcoded the log. What if we want the agent to **read files on its own**?

---

## Step 3: Add a Tool

**Tools** give agents real-world capabilities. `FileReadTool` lets the agent read any file you point it to. We pass the file path as an *input variable* using `{log_file_path}` in the task description.

In [ ]:
from crewai_tools import FileReadTool

log_reader_tool = FileReadTool()

In [ ]:
log_analyzer_with_tool = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    tools=[log_reader_tool],
    verbose=True,
)

analyze_file_task = Task(
    description="""Analyze the log file at {log_file_path} to identify and extract specific issues.
    
    Your analysis should:
    1. Read through the entire log file carefully
    2. Identify all ERROR, CRITICAL, and WARNING messages
    3. Extract the main issue or failure pattern
    4. Determine the timeline of events leading to the failure
    5. Identify the root cause from the log entries""",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer_with_tool,
)

crew = Crew(
    agents=[log_analyzer_with_tool],
    tasks=[analyze_file_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff(inputs={"log_file_path": "../dummy_logs/kubernetes_deployment_error.log"})
print(result.raw)

The agent used `FileReadTool` to read the log file, then analyzed it. With `verbose=True` on the agent, you can see each tool call in the output above.

---

## Step 4: Agent Parameters

Agents can get stuck in loops, burn through API credits, or run forever. These parameters keep them in check.

| Parameter | What It Does |
|-----------|-------------|
| `max_iter` | Max reasoning loops before the agent must produce a final answer |
| `max_rpm` | Rate limit — max LLM requests per minute |
| `max_execution_time` | Hard timeout in seconds |
| `respect_context_window` | Auto-summarize if conversation exceeds the model's context window |

In [ ]:
log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    tools=[log_reader_tool],
    verbose=True,
    max_iter=3,
    max_rpm=10,
    max_execution_time=300,
    respect_context_window=True,
)

print(f"Agent: {log_analyzer.role}")
print(f"  max_iter={log_analyzer.max_iter}, max_rpm={log_analyzer.max_rpm}")
print(f"  max_execution_time={log_analyzer.max_execution_time}s")

---

## Step 5: Second Agent + Search Tool

One agent can only do so much. Our Log Analyzer identifies the problem, but it can't search the internet for solutions. Let's create a second agent with the **EXA Search Tool**.

In [ ]:
from crewai_tools import EXASearchTool

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY")
exa_search_tool = EXASearchTool()

In [ ]:
issue_investigator = Agent(
    role="DevOps Issue Investigator",
    goal="Investigate identified issues by searching documentation, forums, and known solutions online",
    llm=llm,
    backstory="""You are a DevOps troubleshooting specialist who excels at quickly 
    finding solutions to technical problems. You know how to search effectively for 
    similar issues, identify reliable sources, and gather comprehensive information 
    about error patterns and their solutions.""",
    tools=[exa_search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=15,
    max_execution_time=600,
    respect_context_window=True,
)

print(f"Agent created: {issue_investigator.role}")

---

## Step 6: Context Passing

This is the key to multi-agent systems. The `context` parameter on a Task tells CrewAI: *"Before this agent starts, give it the output from these other tasks."*

The Issue Investigator doesn't need to read the log file itself — it receives the Log Analyzer's analysis automatically.

In [ ]:
analyze_task = Task(
    description="""Analyze the log file at {log_file_path} to identify and extract specific issues.
    
    Your analysis should:
    1. Read through the entire log file carefully
    2. Identify all ERROR, CRITICAL, and WARNING messages
    3. Extract the main issue or failure pattern
    4. Determine the timeline of events leading to the failure
    5. Identify the root cause from the log entries""",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

investigate_task = Task(
    description="""Based on the log analysis findings, investigate the identified issue online.
    
    Your investigation should:
    1. Search for similar errors and issues in documentation and forums
    2. Find official documentation related to the error
    3. Look for community solutions and best practices
    4. Identify common causes and scenarios for this type of issue
    5. Gather information about proven fixes and workarounds""",
    expected_output="""A comprehensive investigation report including:
    - Similar issues found online with references
    - Official documentation links
    - Common causes ranked by likelihood
    - Community-verified solutions
    - Best practices to prevent similar issues""",
    agent=issue_investigator,
    context=[analyze_task],
)

crew = Crew(
    agents=[log_analyzer, issue_investigator],
    tasks=[analyze_task, investigate_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff(inputs={"log_file_path": "../dummy_logs/kubernetes_deployment_error.log"})
print(result.raw)

Two agents working together. The Investigator's output references the Analyzer's findings — that's context passing at work.

---

## Step 7: Third Agent — The Synthesizer

The final agent takes the analysis *and* the investigation and produces a concrete remediation plan. Notice `context` can reference **multiple** upstream tasks.

In [ ]:
solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions based on investigation findings",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure and deployment issues. 
    You always provide official documentation references, tested solutions, and 
    preventive measures to avoid future occurrences.""",
    verbose=True,
    max_iter=4,
    max_rpm=8,
    max_execution_time=450,
    respect_context_window=True,
)

print(f"Agent created: {solution_specialist.role}")
print("This agent has no tools — it synthesizes findings from the other two agents.")

---

## Step 8: The Full Pipeline

All three agents. All three tasks. One crew. Everything in one cell so you can see the complete system.

- `Process.sequential` ensures agents run in order.
- `cache=True` avoids redundant LLM calls.
- `max_rpm=30` is the crew-level rate limit.
- `output_file` on each task saves results to disk automatically — useful for audit trails or feeding outputs into other systems.

In [ ]:
os.makedirs("task_outputs", exist_ok=True)

log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    tools=[log_reader_tool],
    verbose=True,
    max_iter=3,
    max_rpm=10,
    max_execution_time=300,
    respect_context_window=True,
)

issue_investigator = Agent(
    role="DevOps Issue Investigator",
    goal="Investigate identified issues by searching documentation, forums, and known solutions online",
    llm=llm,
    backstory="""You are a DevOps troubleshooting specialist who excels at quickly 
    finding solutions to technical problems. You know how to search effectively for 
    similar issues, identify reliable sources, and gather comprehensive information 
    about error patterns and their solutions.""",
    tools=[exa_search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=15,
    max_execution_time=600,
    respect_context_window=True,
)

solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions based on investigation findings",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure and deployment issues. 
    You always provide official documentation references, tested solutions, and 
    preventive measures to avoid future occurrences.""",
    verbose=True,
    max_iter=4,
    max_rpm=8,
    max_execution_time=450,
    respect_context_window=True,
)

analyze_task = Task(
    description="""Analyze the log file at {log_file_path} to identify and extract specific issues.
    
    Your analysis should:
    1. Read through the entire log file carefully
    2. Identify all ERROR, CRITICAL, and WARNING messages
    3. Extract the main issue or failure pattern
    4. Determine the timeline of events leading to the failure
    5. Identify the root cause from the log entries""",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
    output_file="task_outputs/log_analysis.md",
)

investigate_task = Task(
    description="""Based on the log analysis findings, investigate the identified issue online.
    
    Your investigation should:
    1. Search for similar errors and issues in documentation and forums
    2. Find official documentation related to the error
    3. Look for community solutions and best practices
    4. Identify common causes and scenarios for this type of issue
    5. Gather information about proven fixes and workarounds""",
    expected_output="""A comprehensive investigation report including:
    - Similar issues found online with references
    - Official documentation links
    - Common causes ranked by likelihood
    - Community-verified solutions
    - Best practices to prevent similar issues""",
    agent=issue_investigator,
    context=[analyze_task],
    output_file="task_outputs/investigation_report.md",
)

solution_task = Task(
    description="""Based on the log analysis and investigation findings, provide a complete solution.
    
    Your solution should:
    1. Create a step-by-step remediation plan with specific commands
    2. Provide verification steps to confirm the fix
    3. Suggest monitoring and prevention measures
    4. Include rollback procedures if needed
    5. Reference official documentation""",
    expected_output="""A detailed remediation plan with:
    - Primary solution with step-by-step commands
    - Verification and testing procedures
    - Prevention strategies and monitoring recommendations
    - Rollback plan in case of issues
    - Links to official documentation""",
    agent=solution_specialist,
    context=[analyze_task, investigate_task],
    output_file="task_outputs/solution_plan.md",
)

devops_crew = Crew(
    agents=[log_analyzer, issue_investigator, solution_specialist],
    tasks=[analyze_task, investigate_task, solution_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=30,
)

result = devops_crew.kickoff(
    inputs={"log_file_path": "../dummy_logs/kubernetes_deployment_error.log"}
)

In [ ]:
print(result.raw)

---

## Recap

| Step | You Learned | CrewAI API |
|------|------------|------------|
| 1 | Create a specialized agent | `Agent(role, goal, backstory, llm)` |
| 2 | Give it a job | `Task(description, expected_output, agent)` |
| 3 | Give it tools | `tools=[FileReadTool()]`, `{variable}` inputs |
| 4 | Control its behavior | `max_iter`, `max_rpm`, `max_execution_time` |
| 5 | Add more agents | `EXASearchTool()`, multiple agents |
| 6 | Chain tasks together | `context=[upstream_task]` |
| 7 | Synthesize across tasks | `context=[task_1, task_2]` |
| 8 | Run the full pipeline | `Crew(agents, tasks, process, cache, max_rpm)`, `output_file` |

**Next →** Open `02_production_features.ipynb` to add structured output, guardrails, and memory to this pipeline.